# **Извлечение признаков с помощью регулярных выражений**

In [1]:
import pandas as pd
import re
import sklearn
from sklearn.metrics import accuracy_score, f1_score, fbeta_score, mean_absolute_error, precision_score, recall_score
from collections import defaultdict


In [2]:
df = pd.read_csv("df_labeled_all.csv")

In [3]:
print(df.columns)

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'preamble', 'description', 'sentence',
       'killer_count', 'gender_accused', 'prior_convictions', 'alcohol',
       'precrime_argument', 'has_woman_victim', 'has_man_victim', 'method',
       'motive', 'location', 'prison_term'],
      dtype='object')


In [4]:
dff = pd.read_csv("verdicts_with_c.csv")
df_final_100 = pd.read_csv("df_with_marking_final.csv")

In [ ]:
def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()
    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)
    count = 0
    for person in people:
        print(person)
        if 'ст.105' in person:
            count += 1
    print("end")
    return count

df_final_100['killer_count'] = df_final_100['names'].apply(count_killers)


Газизов Дмитрий Рустамович - ст.105 ч.1 УК РФ
end
Жмурко Юрий Геннадьевич - ст.167 ч.1, ст.105 ч.1 УК РФ
end
Скворцов Евгений Владимирович - ст.325 ч.1, ст.325 ч.1, ст.325 ч.2, ст.162 ч.4 п.п.б,в, ст.105 ч.2 п.п.а,ж,з УК РФ
Скородумов Сергей Юрьевич - ст.325 ч.1, ст.325 ч.1, ст.325 ч.2, ст.162 ч.4 п.п.б,в, ст.105 ч.2 п.п.а,ж,з УК РФ
end
Матвеев Сергей Владимирович - ст. 30 ч.3, ст.105 ч.1 УК РФ
end
Шамов Сергей Викторович - ст.105 ч.1 УК РФ
end
Трушковский Юрий Анатольевич - ст.222 ч.1, ст. 30 ч.3, ст.105 ч.2 п.з УК РФ
end
Тарабанов Владимир Викторович - ст.105 ч.2 п.п.а,к УК РФ
end
Топакова Наталья Трофимовна - ст. 30 ч.3, ст.105 ч.1 УК РФ
end
Петрокеев Андрей Анатольевич - ст.105 ч.1 УК РФ
end
Ащеулов Андрей Петрович - ст.162 ч.4 п.в, ст.105 ч.2 п.п.ж,з УК РФ
Барсуков Артём Михайлович - ст.162 ч.4 п.в, ст.105 ч.2 п.п.ж,з УК РФ
end
Ниязгулов Рим Раисович - ст.105 ч.1 УК РФ
end
Лесникова Светлана Александровна - ст.105 ч.1 УК РФ
end
Васильев Сергей Борисович - ст.105 ч.2 п.а УК РФ
end


## **Prison term(срок приговора)**

In [22]:
word2num = {
    "ноль": 0, "один": 1, "одного": 1, "одна": 1, "одной": 1, "одну": 1,
    "два": 2, "двух": 2, "две": 2, "три": 3, "трех": 3, "четыре": 4,
    "пять": 5, "шесть": 6, "семь": 7, "восемь": 8, "восьми": 8,
    "девять": 9, "десять": 10,

    "одиннадцать": 11, "двенадцать": 12, "тринадцать": 13,
    "четырнадцать": 14, "пятнадцать": 15, "шестнадцать": 16,
    "семнадцать": 17, "восемнадцать": 18, "девятнадцать": 19,

    "двадцать": 20, "тридцать": 30, "сорок": 40, "пятьдесят": 50,
    "шестьдесят": 60, "семьдесят": 70, "восемьдесят": 80,
    "девяносто": 90, "сто": 100,

    "полтора": 1.5, "полторы": 1.5, "два с половиной": 2.5,
    "три с половиной": 3.5, "четыре с половиной": 4.5
}

word2num_upper = {k.upper(): v for k, v in word2num.items()}
word2num.update(word2num_upper)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'[«»"\'`]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(
        r'\b(назначить|наказание|срок|лет|год|года|месяц|месяца|месяцев|день|дня|дней)\b',
        lambda m: m.group(0).lower(),
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

def extract_number(value):
    if value is None:
        return 0
    value = str(value).strip()
    if '-' in value:
        parts = value.split('-')
        if all(p.strip().isdigit() for p in parts):
            nums = [int(p.strip()) for p in parts]
            return sum(nums) / len(nums)
    if value.isdigit():
        num = int(value)
        return num if 0 < num <= 100 else 0
    lower_val = value.lower()
    num = word2num.get(lower_val, 0)
    return num if 0 < num <= 100 else 0

def extract_number_with_months(years_str, months_str=None, days_str=None):
    years = extract_number(years_str)
    months = extract_number(months_str) if months_str else 0
    days = extract_number(days_str) if days_str else 0
    if years == 0:
        return 0
    total = years + months/12 + days/365
    return round(min(total, 100), 2)

prison_term_patterns = [

    r"окончательно назначить .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"по совокупности.*?определить\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ]\.\s*[А-ЯЁ]\.\s*наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*([а-я\d\(\)\s]+?) месяц[а-я]*)?",
    r"по совокупности.*?назначить.*?наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)",
    r"путем частичного сложения назначенных наказаний, определить [а-яё]+\s+[а-яё]\.\s*[а-яё]\. наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить .*? наказание в виде лишения свободы на срок ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить .*? наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначив .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)? лишения свободы",
    r"назначить (?:ему|ей) наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы,? сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить наказание в виде лишения свободы сроком ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)",
    r"назначить (?:ему|ей)?\s*наказание\s+([а-яё]+)\s+(?:лет|года|год)\s+лишения свободы",
    r"назначить наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год) лишения свободы",
    r"наказание в виде ограничения свободы на срок ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:год|года|лет)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"наказание .*? ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]* лишения свободы",
    r"лишения свободы на срок ((?:\d+|[а-яё]+)) (?:лет|года|год) ((?:\d+|[а-яё]+)) месяц[а-яё]*",
    r"наказание в виде лишения свободы на срок (\d+)лет\s*(\d+)?\s*месяц[а-я]*",
    r"назначить (?:ему|ей)?\s*наказание\s*(\d+)\s*\([а-я]+\)\s*лет\s*(\d+)?\s*месяц[а-я]*",
    r"наказание в виде\s*(\d+)\s*(?:лет|года|год)\s*лишения свободы",
    r"наказание в виде лишения свободы на срок\s*((?:\d+\s*\([а-яёё\s]+\)))",
    r"в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет лишения свободы",
    r"назначить.*?наказание в виде\s*((?:\d+\s*/[а-яё]+/|[а-яё]+\s*/\d+/|\d+|[а-яё]+))\s*(?:лет|года|год)\s+лишения свободы"

    
    r"(?:назначить|наказание|приговорить|срок)[^.]*?в виде\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s+((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?\s+((?:\d+|[\wЁё]+)\s*(?:день|дня|дней))?",
    r"лишения\s+свободы[^.]*?на\s+срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:день|дня|дней))?",

    r"(?:назначить|наказание)[^.]*?((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s+и\s+((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))",
    r"срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?",

    r"(?:назначить|наказание|приговорить)[^.]*?((?:\d+|[\wЁё]+)\s*(?:лет|года|год))",
    r"лишения\s+свободы[^.]*?на\s+срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))",

    r"В\s+ВИДЕ\s+([А-ЯЁ]+)\s+ЛЕТ\s+([А-ЯЁ]+)?\s*МЕСЯЦ(?:ЕВ|А)?\s*([А-ЯЁ]+)?\s*ДЕН(?:Ь|Я|ЕЙ)?",
    r"НАЗНАЧИТЬ\s+([А-ЯЁ]+)\s+ЛЕТ\s+([А-ЯЁ]+)?\s*МЕСЯЦ(?:ЕВ|А)?",

    r"(?:сроком\s+на|на\s+срок)\s+(\d+)\s*(?:лет|года|год)\s*(?:и\s+)?(\d+)?\s*(?:месяц|месяца|месяцев)?\s*(?:и\s+)?(\d+)?\s*(?:день|дня|дней)?"
]

def extract_prison_term(text):
    try:
        if not text or not isinstance(text, str):
            return None

        text_clean = clean_text(text)
        text_lower = text_clean.lower()
        if any(word in text_lower for word in ["пожизн", "пожизненн"]):
            return 100.0
        for pattern in prison_term_patterns:
            matches = re.finditer(pattern, text_clean, re.IGNORECASE)
            for match in matches:
                groups = match.groups()
                if not groups or not groups[0]:
                    continue

                years = groups[0]
                months = groups[1] if len(groups) > 1 and groups[1] else None
                days = groups[2] if len(groups) > 2 and groups[2] else None
                total = extract_number_with_months(years, months, days)
                if total > 0:
                    return total
        simple_match = re.search(
            r'(?:срок|наказание|назначить)[^.]*?(\d+)\s*(?:лет|года|год)',
            text_clean
        )
        if simple_match:
            return float(simple_match.group(1))

    except Exception as e:
        print(f"Ошибка при обработке текста: {str(e)}")
        return None

    return None

def calculate_metrics(df, true_col='prison_term', pred_col='prison_term_re', thresholds=[0.1, 0.5]):
    df_clean = df.dropna(subset=[true_col, pred_col]).copy()
    df_clean[true_col] = pd.to_numeric(df_clean[true_col], errors='coerce')
    df_clean[pred_col] = pd.to_numeric(df_clean[pred_col], errors='coerce')
    df_clean = df_clean.dropna(subset=[true_col, pred_col])
    y_true = df_clean[true_col]
    y_pred = df_clean[pred_col]
    errors = abs(y_true - y_pred)
    for threshold in thresholds:
        correct = (errors <= threshold).astype(int)

        acc = accuracy_score(correct, [1] * len(correct))
        f1 = f1_score(correct, [1] * len(correct))

        print(f"\nМетрики при пороге {threshold} лет:")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1-score: {f1:.4f}")
        print(f"Доля правильных предсказаний: {correct.mean():.2%}")


In [23]:
df_final_100['prison_term_re'] = df_final_100['sentence'].apply(extract_prison_term)
calculate_metrics(df_final_100, true_col='prison_term', pred_col='prison_term_re', thresholds=[0, 0.5])


Метрики при пороге 0 лет:
Accuracy: 0.8602
F1-score: 0.9249
Доля правильных предсказаний: 86.02%

Метрики при пороге 0.5 лет:
Accuracy: 0.9140
F1-score: 0.9551
Доля правильных предсказаний: 91.40%


In [21]:
print(df_final_100["prison_term"].value_counts())

prison_term
9.00     13
8.00     12
7.00     10
9.50      7
7.50      6
6.50      5
6.00      4
8.50      3
11.00     3
18.00     3
5.00      2
4.00      2
20.00     2
12.00     2
3.00      2
8.08      1
9.25      1
1.00      1
2.50      1
13.83     1
9.83      1
6.92      1
13.00     1
16.00     1
8.75      1
10.00     1
8.67      1
1.67      1
6.60      1
18.50     1
11.50     1
9.91      1
4.83      1
5.50      1
Name: count, dtype: int64


In [20]:
print(df_final_100["prison_term_re"].value_counts())

prison_term_re
9.00       13
7.00       11
8.00       11
9.50        7
6.50        5
7.50        4
20.00       4
6.00        3
8.50        3
11.00       3
3.00        2
5.00        2
4.00        2
13.00       2
18.00       1
8.08        1
9.83        1
6.83        1
9.25        1
1.00        1
10.00       1
2.50        1
13.83       1
1.50        1
10.67       1
8.67        1
2019.00     1
1.67        1
18.50       1
11.50       1
12.00       1
7.25        1
9.92        1
4.83        1
2017.00     1
5.50        1
Name: count, dtype: int64


## **has_woman_victim(была ли среди жертв женщина) и has_man_victim(был ли среди жертв мужчина)**

In [ ]:
woman_patterns = [
    r'\b(потерпевш[ая]|убит[ая]*)\b',
    r'\b(лишив жизни|убийств[оа]?|причинил смерть|напал на)\s+(гражданк[ауе]|девушк[ауе]|женщин[ауе])'
]

man_patterns = [
    r'\b(потерпевш[ий]|убит[ый]|супруг|муж|сын|друг|малолетн[ий]|мальчик)\b',
    r'\b(лишив жизни|убийств[оа]?|причинил смерть|напал на)\s+(гражданин[ауе]|мужчин[ауе])'
]

In [ ]:
'''woman_patterns = [
    r'\b(потерпевш[ая]|убит[ая]|девушка|супруга|жена|дочь|подруга|малолетн[яя]|девочк[а-я]*)\b',
    r'\b(лишив жизни|убийств[оа]?|причинил смерть|напал на)\s+(гражданк[ауе]|девушк[ауе]|женщин[ауе])\b',
    r'\b(умерла|погибла|смерть|загубила|убила)\s+(женщин[ауе])\b',
    r'\b(жертв[а-я]+|погибш[ая]|потерялась|сломала)\b',
    r'\b(в результате нападения на женщину|по отношению к женщине)\b',
    r'\b(вследствие насилия над женщиной|на женщине)\b',
    r'\b(жертва среди женщин|погибла от насилия)\b'
]
man_patterns = [
    r'\b(потерпевш[ий]|убит[ый]|супруг|муж|сын|друг|малолетн[ий]|мальчик)\b',
    r'\b(лишив жизни|убийств[оа]?|причинил смерть|напал на)\s+(гражданин[ауе]|мужчин[ауе])\b',
    r'\b(умер|погиб|погибш[ий]|потерян|умерш[ий])\s+(мужчин[ауе]|парень|муж)\b',
    r'\b(в результате нападения на мужчину|по отношению к мужчине)\b',
    r'\b(жертва среди мужчин|погиб от насилия)\b'
]
'''

def check_victim(text, patterns):
    if not isinstance(text, str):
        return 0
    text = text.lower()
    return int(any(re.search(pat, text) for pat in patterns))

df_final_100["pred_has_woman_victim"] = df_final_100["description"].apply(lambda x: check_victim(x, woman_patterns))
df_final_100["pred_has_man_victim"] = df_final_100["description"].apply(lambda x: check_victim(x, man_patterns))

for col in ["has_woman_victim", "has_man_victim"]:
    y_true = df_final_100[col]
    y_pred = df_final_100["pred_" + col]
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{col} — Accuracy: {acc:.2f}, F1 Score: {f1:.2f}")

errors_woman = df_final_100[df_final_100["has_woman_victim"] != df_final_100["pred_has_woman_victim"]][["description", "has_woman_victim", "pred_has_woman_victim"]]
errors_man = df_final_100[df_final_100["has_man_victim"] != df_final_100["pred_has_man_victim"]][["description", "has_man_victim", "pred_has_man_victim"]]
print("\nОшибки по женщинам:")
print(errors_woman.sample(min(10, len(errors_woman)), random_state=42))

print("\nОшибки по мужчинам:")
print(errors_man.sample(min(10, len(errors_man)), random_state=42))

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\tikho\AppData\Local\Temp\ipykernel_2332\1845710532.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''woman_patterns = [


has_woman_victim — Accuracy: 0.68, F1 Score: 0.11
has_man_victim — Accuracy: 0.26, F1 Score: 0.03

Ошибки по женщинам:
                                          description  has_woman_victim  \
97  У С Т А Н О В И Л:\nБаженов Д.В. совершил убий...                 1   
60  УСТАНОВИЛ:\nВ соответствии с вердиктом коллеги...                 1   
84  УСТАНОВИЛ:\nМихайлова М.С. совершила убийство,...                 1   
68  у с т а н о в и л:\nМихайловский О.В. совершил...                 1   
35  установил:\nФИО9 М.А. совершил убийство, то ес...                 0   
38  УСТАНОВИЛ:\nДанильчук Н.В. совершил убийство, ...                 1   
98  УСТАНОВИЛ:\nМеликян Т.А. совершил убийство, то...                 1   
86  У С Т А Н О В И Л:\nАнаньев Ю.В. совершил убий...                 1   
52  установил:\nШайхиев А.Г. виновен в приготовлен...                 1   
2   УСТАНОВИЛ:\nВердиктом коллегии присяжных засед...                 1   

    pred_has_woman_victim  
97                      0  

## **Motive**

In [ ]:
motive_mapping = {
    'личная неприязнь': 'личная неприязнь',
    'неприязнь': 'личная неприязнь',
    'ссора': 'личная неприязнь',
    'обида': 'личная неприязнь',
    'злость': 'личная неприязнь',
    'агрессия': 'личная неприязнь',
    'ярость': 'личная неприязнь',
    'раздражение': 'личная неприязнь',
    'оскорбление': 'личная неприязнь',
    'ненависть': 'личная неприязнь',

    'ревность': 'ревность',
    'зависть': 'ревность',

    'месть': 'месть',

    'корысть': 'корысть',
    'имущество': 'корысть',
    'ограбление': 'корысть',
    'хищение': 'корысть',
    'кража': 'корысть',
    'деньги': 'корысть',
    'личное обогащение': 'корысть',
    'наследство': 'корысть',
    'вымогательство': 'корысть',

    'алкоголь': 'алкоголь/наркотики',
    'алкогольное опьянение': 'алкоголь/наркотики',
    'наркотики': 'алкоголь/наркотики',

    'аморальное поведение': 'аморальность',
    'аморальность': 'аморальность',
    'противоправное поведение': 'аморальность',
    'унижение': 'аморальность',
    'оскорбления': 'аморальность',

    'насилие': 'насилие',
    'половое насилие': 'насилие',
    'сексуальное домогательство': 'насилие',
    'попытка изнасилования': 'насилие',

    'неизвестно': 'прочее',
    'сострадание': 'прочее',
    'страх': 'прочее',
    'жалость': 'прочее',
    'самозащита': 'прочее',
    'защита': 'прочее',
}

category_keywords = defaultdict(list)
for keyword, group in motive_mapping.items():
    category_keywords[group].append(re.escape(keyword))

motive_patterns = {
    group: r'\b(' + '|'.join(keywords) + r')\b'
    for group, keywords in category_keywords.items()
}

In [ ]:
motive_patterns = {
    "личная неприязнь": r'\b(личная неприязнь|на почве личной неприязни|в результате ссоры|в ходе ссоры|ссора|обида|затаённая обида|злость|агрессия|ярость|раздражение|оскорбление(?:ми)?|в ответ на оскорбление|ненависть)\b',

    "ревность": r'\b(ревность|на почве ревности|вспышка ревности|зависть)\b',

    "корысть": r'\b(ограбление|имущество|хищение|кража|деньги|обогащение|личное обогащение|наследство|вымогательство|корыстн\w*|с целью наживы|в корыстных целях|из корыстных побуждений)\b',

    "месть": r'\b(месть|в отместку|из чувства мести|в качестве мести|на почве мести)\b',

    "алкоголь/наркотики": r'\b(в состоянии алкогольного опьянения|алкогольное опьянение|алкоголь|употребив алкоголь|наркотик\w*|в состоянии наркотического опьянения|в состоянии опьянения)\b',

    "аморальность": r'\b(аморальное поведение|аморальность|противоправное поведение|девиантное поведение|унижение|моральное унижение|оскорбления|оскорбительное поведение)\b',

    "насилие": r'\b(половое насилие|сексуальное домогательство|изнасилование|попытка изнасилования|насильственные действия|принуждение к половой связи)\b',

    "прочее": r'\b(сострадание|страх|испуг|паника|жалость|в целях самозащиты|самозащита|защита|неизвестно|не установлен)\b',
}

def extract_motive(text):
    if not isinstance(text, str):
        return "прочее"
    text = text.lower()
    for group, pattern in motive_patterns.items():
        if re.search(pattern, text):
            return group
    return "прочее"

def map_single_motive(text):
    text = str(text).lower()
    for keyword, group in motive_mapping.items():
        if keyword in text:
            return group
    return "прочее"

df_final_100["predicted_motive"] = df_final_100["description"].apply(extract_motive)
df_final_100["true_motive"] = df_final_100["motive"].apply(map_single_motive)

accuracy = accuracy_score(df_final_100["true_motive"], df_final_100["predicted_motive"])
f1 = f1_score(df_final_100["true_motive"], df_final_100["predicted_motive"], average='micro')

print(f"Accuracy: {accuracy:.2f}")
print(f"F1 Score: {f1:.2f}")

errors = df_final_100[df_final_100["true_motive"] != df_final_100["predicted_motive"]][["description", "true_motive", "predicted_motive"]]
print("\nОшибки предсказания (пример):")
print(errors.sample(min(10, len(errors)), random_state=42))


Accuracy: 0.68
F1 Score: 0.68

Ошибки предсказания (пример):
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
print(dff["motive"].value_counts())
print(dff["motive_group"].value_counts())

motive
личная неприязнь              76
ревность                       7
прочее                         4
личная неприязнь, ревность     4
личная неприязнь, месть        3
корысть                        2
корысть, личная неприязнь      2
корысть, месть                 1
месть                          1
Name: count, dtype: int64
motive_group
личная неприязнь                                                        14
аморальность, личная неприязнь                                          11
аморальность, корысть, личная неприязнь                                  8
корысть                                                                  8
прочее                                                                   7
алкоголь/наркотики, аморальность, личная неприязнь                       6
алкоголь/наркотики, корысть, личная неприязнь                            5
аморальность                                                             5
аморальность, личная неприязнь, насилие                  

## **Method**

In [ ]:
method_map = {
    'избиение': 'физическое воздействие', 'удар предметом': 'физическое воздействие',
    'скидывание с высоты': 'физическое воздействие',
    'нож': 'холодное оружие', 'клинок': 'холодное оружие', 'вилка': 'холодное оружие',
    'лопата': 'холодное оружие', 'топор': 'холодное оружие',
    'молоток': 'тяжелые предметы', 'кирпич': 'тяжелые предметы', 'полено': 'тяжелые предметы',
    'ружье': 'огнестрельное оружие', 'травматический пистолет': 'огнестрельное оружие',
    'огнестрел': 'огнестрельное оружие',
    'удушение': 'удушение', 'повешение': 'удушение'
}
def categorize_method(method):
    return method_map.get(method, 'неизвестно')

df_final_100["categorized_method"] = df_final_100["method"].apply(categorize_method)

method_patterns = {
    'физическое воздействие': r'\b(избиение|удар предметом|скидывание с высоты)\b',
    'холодное оружие': r'\b(нож|клинок|вилка|лопата|топор)\b',
    'тяжелые предметы': r'\b(молоток|кирпич|полено)\b',
    'огнестрельное оружие': r'\b(ружье|травматический пистолет|огнестрел)\b',
    'удушение': r'\b(удушение|повешение)\b'
}

def extract_method(text):
    if not isinstance(text, str):
        return "неизвестно"
    for method, pattern in method_patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return method
    return "неизвестно"

df_final_100["predicted_method"] = df_final_100["description"].apply(extract_method)

accuracy = accuracy_score(df_final_100["categorized_method"], df_final_100["predicted_method"])
f1 = f1_score(df_final_100["categorized_method"], df_final_100["predicted_method"], average='micro')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")


Accuracy: 0.63
F1 Score: 0.63


In [ ]:
print(df_final_100["method"].value_counts())

method
нож                                                      55
избиение                                                 10
удушение                                                  6
огнестрел                                                 4
молоток                                                   4
топор                                                     4
ружье                                                     1
удар предметом                                            1
травматический пистолет                                   1
полено                                                    1
клинок                                                    1
['избиение', 'удушение']                                  1
['избиение', 'молоток']                                   1
['нож', 'удушение']                                       1
['нож', 'избиение']                                       1
['удушение', 'избиение']                                  1
['молоток', 'топор']             

In [ ]:
print(df_final_100["predicted_method"].value_counts())

predicted_method
холодное оружие           68
неизвестно                14
физическое воздействие     7
тяжелые предметы           5
огнестрельное оружие       4
удушение                   2
Name: count, dtype: int64


## **Location**

In [ ]:
def categorize_location(location):
    location_map = {
        'квартира': 'жилое помещение', 'дом': 'жилое помещение', 'дача': 'жилое помещение',
        'общежитие': 'жилое помещение', 'хостел': 'жилое помещение',
        'улица': 'уличное пространство', 'двор': 'уличное пространство',
        'безлюдная местность': 'уличное пространство', 'берег озера': 'уличное пространство',
        'лес': 'уличное пространство',
        'такси': 'общественный транспорт', 'автобус': 'общественный транспорт',
        'поезд': 'общественный транспорт',
        'магазин': 'общественное место', 'кафе': 'общественное место',
        'бар': 'общественное место', 'рынок': 'общественное место',
        'подъезд': 'подъезд и прилегающие зоны', 'лестничная площадка': 'подъезд и прилегающие зоны',
        'чердак': 'подъезд и прилегающие зоны',
        'гараж': 'рабочая зона', 'база отдыха': 'рабочая зона', 'склад': 'рабочая зона',
        'офис': 'рабочая зона'
    }
    return location_map.get(location, 'неизвестно')

df_final_100["categorized_location"] = df_final_100["location"].apply(categorize_location)


In [ ]:
location_patterns = {
    'жилое помещение': r'\b(квартира|квартир|квартире|квартирой|дом|дома|доме|домом|дача|дачи|дачей|общежитие|общежития|общежитием|хостел|хостела|хостеле)\b',
    'уличное пространство': r'\b(улица|улицы|улице|улицей|двор|дворе|двором|безлюдная местность|безлюдной местности|берег озера|берега озера|берегу озера|лес|лесу|лесом)\b',
    'общественный транспорт': r'\b(такси|такси|автобус|автобусе|автобусом|поезд|поезде|поездом)\b',
    'общественное место': r'\b(магазин|магазина|магазине|магазином|кафе|кафе|бар|бара|баре|баром|рынок|рынка|рынке|рынком)\b',
    'подъезд и прилегающие зоны': r'\b(подъезд|подъезда|подъезде|подъездом|лестничная площадка|лестничной площадки|лестничной площадке|лестничной площадкой|чердак|чердака|чердаке|чердаком)\b',
    'рабочая зона': r'\b(гараж|гаража|гараже|гаражом|база отдыха|базы отдыха|базе отдыха|базой отдыха|склад|склада|складе|складом|офис|офиса|офисе|офисом)\b'
}

def extract_location(text):
    for category, pattern in location_patterns.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            return category
    return 'неизвестно'

df_final_100["predicted_location"] = df_final_100["description"].apply(extract_location)

accuracy = accuracy_score(df_final_100["categorized_location"], df_final_100["predicted_location"])
f1 = f1_score(df_final_100["categorized_location"], df_final_100["predicted_location"], average='micro')

print(f"Accuracy: {accuracy}")
print(f"F1-score (micro): {f1}")


Accuracy: 0.77
F1-score (micro): 0.77


In [ ]:
mismatches = df_final_100[
    (df_final_100["categorized_location"] != df_final_100["predicted_location"])
]
print(len(mismatches))
print(mismatches[["id", "categorized_location", "predicted_location"]])


23
        id        categorized_location predicted_location
0    57844        уличное пространство    жилое помещение
1    71745        уличное пространство    жилое помещение
5    64171        уличное пространство    жилое помещение
9    12555                  неизвестно    жилое помещение
14   16294        уличное пространство    жилое помещение
15    7954        уличное пространство    жилое помещение
17   81538                рабочая зона    жилое помещение
33  117915        уличное пространство    жилое помещение
35  103663        уличное пространство    жилое помещение
41  105000  подъезд и прилегающие зоны    жилое помещение
43   66092        уличное пространство    жилое помещение
45   71133        уличное пространство    жилое помещение
48   43146        уличное пространство    жилое помещение
52   34394        уличное пространство    жилое помещение
64   83661  подъезд и прилегающие зоны    жилое помещение
75   99251        уличное пространство    жилое помещение
80   53901 

## **Relationship**

In [ ]:
df_final_100["relationship_between_accused_and_victim"].value_counts()

relationship_between_accused_and_victim
знакомый                        45
сожитель                        17
супруг                           7
незнакомец                       6
друг                             4
родственник                      3
коллега                          3
интимные отношения               2
брат                             2
зять                             2
сосед                            2
неизвестно                       1
['неизвестно', 'неизвестно']     1
мать сожительницы                1
['знакомый', 'знакомый']         1
работник                         1
['сын', 'знакомый']              1
бывший супруг                    1
Name: count, dtype: int64

Группируем по категориям

In [ ]:
group_map = {
    "супруг": "супруг/сожитель",
    "сожитель": "супруг/сожитель",
    "сожительница": "супруг/сожитель",
    "гражданский муж": "супруг/сожитель",
    "бывший супруг": "супруг/сожитель",
    "бывшая сожительница": "супруг/сожитель",
    "бывший молодой человек": "супруг/сожитель",
    "гражданский": "супруг/сожитель",
    "любовник": "супруг/сожитель",

    "брат": "близкий родственник",
    "сестра": "близкий родственник",
    "отец": "близкий родственник",
    "мать": "близкий родственник",
    "матерь": "близкий родственник",
    "родитель": "близкий родственник",
    "сын": "близкий родственник",
    "дочь": "близкий родственник",
    "близкий родственник": "близкий родственник",

    "дядя": "не близкий родственник",
    "дядь": "не близкий родственник",
    "тётя": "не близкий родственник",
    "племянник": "не близкий родственник",
    "бабушка": "не близкий родственник",
    "внук": "не близкий родственник",
    "зять": "не близкий родственник",
    "свекор": "не близкий родственник",
    "свекровь": "не близкий родственник",
    "отчим": "не близкий родственник",
    "падчерица": "не близкий родственник",
    "пасынок": "не близкий родственник",
    "приемный сын": "не близкий родственник",
    "двоюродный брат": "не близкий родственник",
    "троюродный брат": "не близкий родственник",
    "сводный брат": "не близкий родственник",
    "двоюродный племянник": "не близкий родственник",
    "сводная бабушка": "не близкий родственник",
    "родственник": "не близкий родственник",
    "дальний родственник": "не близкий родственник",
    "дальние родственник": "не близкий родственник",
    "близкий родственник": "не близкий родственник",

    "знакомый": "знакомый",
    "знакомая": "знакомый",
    "приятель": "знакомый",
    "товарищ": "знакомый",
    "подруга": "знакомый",

    "незнакомец": "незнакомец",
    "незнакомый": "незнакомец",
    "незнакомка": "незнакомец",

    "друг": "друг",
    "подруга": "друг",
    "приятель": "друг",
    "товарищ": "друг",

    "сосед": "сосед",
    "соседка": "сосед",

    "работник": "прочее",
    "работодатель": "прочее",
    "водитель": "прочее",
    "сокамерник": "прочее",
    "одноклассник": "прочее",

    "неизвестно": "неизвестно"
}

df_final_100["true_relationship_grouped"] = df_final_100["relationship_between_accused_and_victim"].map(group_map).fillna("прочее")


In [ ]:
df_final_100.loc[:, "predicted_relationship_grouped"] = df_final_100["predicted_relationship_re"].map(group_map).fillna("прочее")
df_final_100["predicted_relationship_grouped"].value_counts()

In [ ]:
accuracy = accuracy_score(
    df_final_100["true_relationship_grouped"],
    df_final_100["predicted_relationship_re"]
)
f1 = f1_score(
    df_final_100["true_relationship_grouped"],
    df_final_100["predicted_relationship_re"],
    average='micro'
)
print(f"Accuracy: {accuracy:.3f}")
print(f"F1 (micro): {f1:.3f}")


Accuracy: 0.370
F1 (micro): 0.370


In [ ]:
mismatches = df_final_100[
    (df_final_100["true_relationship_grouped"] != df_final_100["predicted_relationship_re"])
]
print(len(mismatches))
print(mismatches[["id", "true_relationship_grouped", "predicted_relationship_re"]])


63
        id true_relationship_grouped predicted_relationship_re
0    57844                  знакомый                незнакомец
2    41579                неизвестно                незнакомец
3   116462                    прочее                  знакомый
4   102232                  знакомый                незнакомец
6    87680                    прочее                незнакомец
..     ...                       ...                       ...
91  130362           супруг/сожитель                  знакомый
94   92323           супруг/сожитель                  знакомый
96   79171                  знакомый       близкий родственник
97   31607                    прочее                  знакомый
99    2220                незнакомец    не близкий родственник

[63 rows x 3 columns]


In [ ]:
print(df_final_100[["true_relationship_grouped","id","predicted_relationship_re"]])

   true_relationship_grouped      id predicted_relationship_re
0                   знакомый   57844                  знакомый
1                   знакомый   71745           супруг/сожитель
2                 неизвестно   41579                незнакомец
3                     прочее  116462           супруг/сожитель
4                   знакомый  102232                  знакомый
..                       ...     ...                       ...
95                  знакомый  122495                  знакомый
96                  знакомый   79171       близкий родственник
97                    прочее   31607       близкий родственник
98           супруг/сожитель   14640           супруг/сожитель
99                незнакомец    2220    не близкий родственник

[100 rows x 3 columns]


In [ ]:
print(df_final_100["true_relationship_grouped"].value_counts())

true_relationship_grouped
знакомый                  45
супруг/сожитель           25
прочее                    10
незнакомец                 6
не близкий родственник     5
друг                       4
близкий родственник        2
сосед                      2
неизвестно                 1
Name: count, dtype: int64


In [ ]:
print(df_final_100["true_relationship_grouped"].value_counts())

true_relationship_grouped
знакомый                  45
супруг/сожитель           25
прочее                    10
незнакомец                 6
не близкий родственник     5
друг                       4
близкий родственник        2
сосед                      2
неизвестно                 1
Name: count, dtype: int64


In [ ]:
relation_patterns = {
    "супруг/сожитель": r"\b(супруг|сожительница?|гражданский муж|бывш[аяий] (супруг|сожительница?|молодой человек)|любовник)\b",
    "близкий родственник": r"\b(брат|сестра|отец|мать|матерь|родитель|сын|дочь)\b",
    "не близкий родственник": r"\b(дяд[яю]|тётя|племянник|бабушка|внук|зять|свекро(вь|м)|отчим|падчерица|пасынок|приемный сын|двоюродный|троюродный|сводный|родственник|дальний|сводная бабушка)\b",
    "знакомый": r"\b(знаком(ый|ая)|приятель|подруга|товарищ|гости)\b",
    "незнакомец": r"\b(незнаком(ец|ка|ый))\b",
    "друг": r"\b(?<!друг )\bдруг(?! друга|их|им|у|ом)\b",
    "сосед": r"\b(сосед(ка)?|по соседству)\b",
    "прочее": r"\b(работник|работодатель|водитель|сокамерник|одноклассник)\b"
}

def extract_relationship(text):
    if not isinstance(text, str):
        return "незнакомец"
    if re.search(r"\bдруг друга\b", text.lower()):
        text = re.sub(r"\bдруг друга\b", "", text, flags=re.IGNORECASE)
    for category in [
        "знакомый",
        "супруг/сожитель",
        "незнакомец",
        "не близкий родственник",
        "близкий родственник",
        "друг",
        "сосед",
        "прочее"
    ]:
        pattern = relation_patterns[category]
        if re.search(pattern, text, flags=re.IGNORECASE):
            return category

    return "незнакомец"

df_final_100["predicted_relationship_re"] = df_final_100["description"].apply(extract_relationship)


In [ ]:
print(df_final_100["predicted_relationship_re"].value_counts())

predicted_relationship_re
знакомый                  60
близкий родственник       14
незнакомец                13
супруг/сожитель            6
не близкий родственник     5
друг                       2
Name: count, dtype: int64


In [ ]:
print(df_final_100[["predicted_relationship_re", "id", "relationship_between_accused_and_victim"]])

   predicted_relationship_re      id relationship_between_accused_and_victim
0                       друг   57844                                знакомый
1            супруг/сожитель   71745                                знакомый
2                 неизвестно   41579                              неизвестно
3            супруг/сожитель  116462                                работник
4                       друг  102232                                знакомый
..                       ...     ...                                     ...
95                  знакомый  122495                                знакомый
96       близкий родственник   79171                                знакомый
97       близкий родственник   31607                      интимные отношения
98           супруг/сожитель   14640                                сожитель
99       близкий родственник    2220                              незнакомец

[100 rows x 3 columns]


In [ ]:
print(df_final_100["relationship_between_accused_and_victim"].value_counts())

relationship_between_accused_and_victim
знакомый                        45
сожитель                        17
супруг                           7
незнакомец                       6
друг                             4
родственник                      3
коллега                          3
интимные отношения               2
брат                             2
зять                             2
сосед                            2
неизвестно                       1
['неизвестно', 'неизвестно']     1
мать сожительницы                1
['знакомый', 'знакомый']         1
работник                         1
['сын', 'знакомый']              1
бывший супруг                    1
Name: count, dtype: int64


## **Gender_accused**

In [ ]:
def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()

    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)

    count = 0
    for person in people:
        print(person)
        if 'ст.105' in person:
            count += 1
    print("end")
    return count

In [ ]:
male_patterns = [re.compile(p, re.IGNORECASE) for p in [
    r"(подсудимый|обвиняемый|осужденного|обвиняемого|виновный|виновного)",
    r"(убил|избил|ранил|нанес|совершил)",
]]

female_patterns = [re.compile(p, re.IGNORECASE) for p in [
    r"(подсудимая|обвиняемая|осужденная)",
    r"(убила|избила|ранила|нанесла|совершила)",
]]

def extract_gender_combined(row):
    combined_text = f"{row['preamble']} {row['description']}".lower()
    killer_gender = "неизвестно"
    victim_gender = "неизвестно"
    for pattern in male_patterns:
        if pattern.search(combined_text):
            killer_gender = "мужчина"
            break
    for pattern in female_patterns:
        if pattern.search(combined_text):
            killer_gender = "женщина"
            break
    if re.search(r"\bпотерпевший\b", combined_text):
        victim_gender = "мужчина"
    elif re.search(r"\bпотерпевшая\b", combined_text):
        victim_gender = "женщина"
    return pd.Series([killer_gender, victim_gender])

df_final_100[["gender_accused", "gender_victim"]] = df_final_100.apply(extract_gender_combined, axis=1)


In [ ]:
male_patterns = [re.compile(p, re.IGNORECASE) for p in [
    r"(подсудимый|обвиняемый|осужденного|обвиняемого|виновный|виновный)",
    r"(убил|избил|ранил|нанес|совершил)",
]]

female_patterns = [re.compile(p, re.IGNORECASE) for p in [
    r"(подсудимая|обвиняемая|осужденная)",
    r"(убила|избила|ранила|нанесла|совершила)",
]]

def extract_gender_combined(row):
    combined_text = f"{row['preamble']} {row['description']}".lower()
    killer_gender = "неизвестно"
    victim_gender = "неизвестно"
    for pattern in male_patterns:
        if pattern.search(combined_text):
            killer_gender = "мужчина"
            break
    for pattern in female_patterns:
        if pattern.search(combined_text):
            killer_gender = "женщина"
            break
    if re.search(r"\bпотерпевший\b", combined_text):
        victim_gender = "мужчина"
    elif re.search(r"\bпотерпевшая\b", combined_text):
        victim_gender = "женщина"

    return pd.Series([killer_gender, victim_gender])

df_final_100[["predicted_killer_gender", "predicted_victim_gender"]] = df_final_100.apply(extract_gender_combined, axis=1)

def compute_metrics(y_true, y_pred, label):
    print(f"\n Метрики для: {label}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.2f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"F1-score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")

df_killer_cleaned = df_final_100[
    (df_final_100["killer_count"] == 1) &
    (df_final_100["gender_accused"].isin(["мужчина", "женщина"])) &
    (df_final_100["predicted_killer_gender"].isin(["мужчина", "женщина"]))
]

df_victim_cleaned = df_final_100[
    (df_final_100["gender_victim"].isin(["мужчина", "женщина"])) &
    (df_final_100["predicted_victim_gender"].isin(["мужчина", "женщина"]))
]

compute_metrics(df_killer_cleaned["gender_accused"], df_killer_cleaned["predicted_killer_gender"], "Пол обвиняемого (killer_count=1)")
compute_metrics(df_victim_cleaned["gender_victim"], df_victim_cleaned["predicted_victim_gender"], "Пол жертвы")


In [ ]:
df["gender_accused"] = df["gender_accused"].apply(lambda x: x[0] if isinstance(x, list) else x)
df["predicted_killer_gender"] = df["predicted_killer_gender"].apply(lambda x: x[0] if isinstance(x, list) else x)

df["gender_victim"] = df["gender_victim"].apply(lambda x: x[0] if isinstance(x, list) else x)
df["predicted_victim_gender"] = df["predicted_victim_gender"].apply(lambda x: x[0] if isinstance(x, list) else x)

mask_killer = df["gender_accused"].notna() & df["predicted_killer_gender"].notna()
mask_victim = df["gender_victim"].notna() & df["predicted_victim_gender"].notna()

y_true_killer = df.loc[mask_killer, "gender_accused"]
y_pred_killer = df.loc[mask_killer, "predicted_killer_gender"]

y_true_victim = df.loc[mask_victim, "gender_victim"]
y_pred_victim = df.loc[mask_victim, "predicted_victim_gender"]

def print_metrics(y_true, y_pred, label):
    print(f"\n Метрики для {label}:")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.2f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"F1-score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")

print_metrics(y_true_killer, y_pred_killer, "Пола убийцы")
print_metrics(y_true_victim, y_pred_victim, "Пола жертвы")



📊 Метрики для Пола убийцы:
Accuracy:  0.80
Precision: 0.64
Recall:    0.80
F1-score:  0.71

📊 Метрики для Пола жертвы:
Accuracy:  0.69
Precision: 0.72
Recall:    0.69
F1-score:  0.70


## **Drugs**

In [ ]:

narcotic_patterns = [
    r"(в состоянии\s+наркотического\s+опьянения)",
    r"(находясь\s+под\s+влиянием\s+наркотиков)",
    r"(находился\s+в\s+состоянии\s+наркотического\s+опьянения)",
    r"(употребив\s+(наркотик|наркотики|психоактивное\s+вещество))",
    r"(под\s+действием\s+наркотик\w*)",
    r"(влиянием\s+психотропных\s+веществ)",
    r"(был\s+обнаружен\s+наркотик\w*)",
    r"(в крови\s+обнаружены\s+следы\s+наркотик\w*)"
]
df_final_100["drugs"] = df_final_100["drugs"].astype(str).str.strip().str.lower().replace(["-", "nan", "—"], "нет")
def check_narcotics(text):
    for pattern in narcotic_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return "да"
    return "нет"
df_final_100["predicted_narcotics"] = df_final_100["description"].apply(check_narcotics)
compute_metrics(df_final_100["drugs"], df_final_100["predicted_narcotics"], "Наркотического опьянения")

KeyError: 'drugs'

## **Alcohol**

In [ ]:
alcohol_patterns = [
    r"(в состоянии\s+алкогольного\s+опьянения)",
    r"(алкогольное\s+опьянение)",
    r"(находясь\s+в\s+состоянии\s+опьянения)",
    r"(находился\s+в\s+состоянии\s+алкогольного\s+опьянения)",
    r"(выпив\s+алкоголь|выпив\s+спиртное)",
    r"(употребив\s+(алкоголь|спиртное))",
    r"(был\s+пьян|была\s+пьяна)",
    r"(выпивал\s+(водку|пиво|спиртное))",
    r"(влиянием\s+алкоголя)",
    r"(опьянение,\s+вызванное\s+употреблением\s+алкоголя)",
]

def check_alcohol(text):
    for pattern in alcohol_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return "да"
    return "нет"

df_final_100['alcohol'] = df_final_100['description'].apply(check_alcohol)
df_final_100['alcohol_predicted'] = df_final_100["description"].apply(check_alcohol)


In [ ]:
print(df['alcohol'].value_counts())

alcohol
да     29274
нет     5491
Name: count, dtype: int64


In [ ]:
y_true = df_final_100['alcohol'].apply(lambda x: 1 if x == "да" else 0)
y_pred = df_final_100['alcohol_predicted'].apply(lambda x: 1 if x == "да" else 0)
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Accuracy: 0.8800
Precision: 0.8929
Recall: 0.9615
F1-score: 0.9259


In [ ]:
multiple_named = dff[dff['killer_count'] >= 2]
if not multiple_named.empty:
    print("Строки с двумя и более осужденными:")
    print(multiple_named[['relationship_between_accused_and_victim', 'killer_count','id']].to_string(index=False))
else:
    print("Нет строк с двумя и более осужденными")

Строки с двумя и более осужденными:
relationship_between_accused_and_victim  killer_count     id
           ['неизвестно', 'неизвестно']             2  41579
           ['незнакомец', 'незнакомец']             2  12555
              ['коллега', 'незнакомец']             2 105000


In [ ]:
import re

word2num = {
    "ноль": 0, "один": 1, "одного": 1, "одна": 1, "два": 2, "двух": 2, "две": 2,
    "три": 3, "трех": 3, "четыре": 4, "пять": 5, "шесть": 6, "семь": 7,
    "восемь": 8, "восьми": 8, "девять": 9, "десять": 10,
    "одиннадцать": 11, "двенадцать": 12, "тринадцать": 13, "четырнадцать": 14,
    "пятнадцать": 15, "шестнадцать": 16, "семнадцать": 17, "восемнадцать": 18,
    "девятнадцать": 19, "двадцать": 20, "тридцать": 30, "сорок": 40,
    "пятьдесят": 50, "шестьдесят": 60, "семьдесят": 70,
    "восемьдесят": 80, "девяносто": 90, "сто": 100
}

def extract_number(value):
    if value is None:
        return 0
    value = value.strip().lower()
    if value.isdigit():
        return int(value)
    return word2num.get(value, 0)

def extract_number_with_months(years_str, months_str=None):
    years = extract_number(years_str)
    months = extract_number(months_str) if months_str else 0
    return years + months / 12

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

prison_term_patterns = [
    r"(?:назначить|приговорить|наказание[^.]*?)\s*(?:[^.]{0,40}?)?(?:в виде)?[^.]{0,40}?(?:лишения\s+свободы)?[^.]{0,40}?(?:на срок)?\s*([а-яё]+)\s+лет(?:\s+и\s+([а-яё]+)\s+месяц)?",
    r"(?:назначить|приговорить|наказание[^.]*?)\s*(?:[^.]{0,40}?)?(?:в виде)?[^.]{0,40}?(?:лишения\s+свободы)?[^.]{0,40}?(?:на срок)?\s*(\d+)[^0-9а-яё]{0,10}(?:лет|года|год)(?:[^0-9а-яё]{0,10}(\d+))?"
]

def extract_prison_term(text):
    text = clean_text(text)
    if "пожизн" in text:
        return 100.0
    text = re.split(r"лишени[ея]\s+права\s+заниматься", text)[0]

    for pattern in prison_term_patterns:
        for match in re.finditer(pattern, text):
            groups = match.groups()
            if len(groups) >= 1:
                years = groups[0]
                months = groups[1] if len(groups) > 1 else None
                total = extract_number_with_months(years, months)
                if total > 0:
                    return round(total, 2)
    return None


In [ ]:
def calculate_extended_metrics(df, threshold=0.5):
    df_clean = df.dropna(subset=['prison_term', 'prison_term_re']).copy()
    df_clean['prison_term'] = pd.to_numeric(df_clean['prison_term'], errors='coerce')
    df_clean['prison_term_re'] = pd.to_numeric(df_clean['prison_term_re'], errors='coerce')
    df_clean = df_clean.dropna(subset=['prison_term', 'prison_term_re'])

    y_true = df_clean['prison_term']
    y_pred = df_clean['prison_term_re']
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.2f} лет")
    print(f"R²: {r2_score(y_true, y_pred):.3f}")
    accurate = (y_true - y_pred).abs() <= threshold
    accuracy = accuracy_score(accurate, [True]*len(accurate))
    print(f"\nAccuracy (разница ≤ {threshold} лет): {accuracy:.2%}")
    y_true_binary = ((y_true - y_pred).abs() > threshold).astype(int)
    y_pred_binary = [0]*len(y_true_binary)
    f1 = f1_score(y_true_binary, y_pred_binary, pos_label=0)
    print(f"F1-score (для разницы ≤ {threshold} лет): {f1:.3f}")
    print("\nДОПОЛНИТЕЛЬНАЯ СТАТИСТИКА:")
    exact_matches = (y_true.round(1) == y_pred.round(1)).mean()
    print(f"Точные совпадения (до 0.1 года): {exact_matches:.2%}")
    diff = (y_pred - y_true).abs()
    print(f"Средняя разница: {diff.mean():.2f} ± {diff.std():.2f} лет")
    print(f"Медианная разница: {diff.median():.2f} лет")
    print("\nРАСПРЕДЕЛЕНИЕ ОШИБОК:")
    print(diff.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))
    print("\nТОП-5 НАИБОЛЬШИХ РАСХОЖДЕНИЙ:")
    top_errors = df_clean.iloc[diff.nlargest(5).index]
    print(top_errors[['id','prison_term', 'prison_term_re', 'sentence']].to_string(index=False))
calculate_extended_metrics(dff, threshold=1.0)


РЕГРЕССИОННЫЕ МЕТРИКИ:
MAE: 909.88 лет
R²: -147455.914

Accuracy (разница ≤ 1.0 лет): 50.53%
F1-score (для разницы ≤ 1.0 лет): 0.671

ДОПОЛНИТЕЛЬНАЯ СТАТИСТИКА:
Точные совпадения (до 0.1 года): 26.32%
Средняя разница: 909.88 ± 1005.48 лет
Медианная разница: 1.00 лет

РАСПРЕДЕЛЕНИЕ ОШИБОК:
count      95.000000
mean      909.882526
std      1005.475407
min         0.000000
25%         0.000000
50%         1.000000
75%      2010.000000
90%      2013.300000
95%      2014.206000
max      2019.330000
dtype: float64

ТОП-5 НАИБОЛЬШИХ РАСХОЖДЕНИЙ:
    id  prison_term  prison_term_re                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [ ]:
word2num = {
    "ноль": 0, "один": 1, "одного": 1, "одна": 1, "два": 2, "двух": 2, "две": 2,
    "три": 3, "трех": 3, "четыре": 4, "пять": 5, "шесть": 6, "семь": 7,
    "восемь": 8, "восьми": 8, "девять": 9, "десять": 10,
    "одиннадцать": 11, "двенадцать": 12, "тринадцать": 13, "четырнадцать": 14,
    "пятнадцать": 15, "шестнадцать": 16, "семнадцать": 17, "восемнадцать": 18,
    "девятнадцать": 19, "двадцать": 20, "тридцать": 30, "сорок": 40,
    "пятьдесят": 50, "шестьдесят": 60, "семьдесят": 70,
    "восемьдесят": 80, "девяносто": 90, "сто": 100
}

word2num_upper = {k.upper(): v for k, v in word2num.items()}
word2num.update(word2num_upper)

def extract_number(value):
    """Преобразует строковое представление числа (слово или цифры) в int"""
    if value is None:
        return 0
    value = value.strip()
    if value.isdigit():
        return int(value)
    return word2num.get(value.lower(), word2num.get(value, 0))

def extract_number_with_months(years_str, months_str=None, days_str=None):
    years = extract_number(years_str)
    months = extract_number(months_str) if months_str else 0
    days = extract_number(days_str) if days_str else 0
    return years + months / 12 + days / 365

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

prison_term_patterns = [
    r"наказание[^.]*?в виде\s+([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+дней",
    r"назначить[^.]*?([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+дней",
    r"срок\s+([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+дней",
    r"([а-яёА-ЯЁ]+)\s+лет\s+([а-яёА-ЯЁ]+)\s",
    r"(\d+)\s+лет\s+(\d+)\s+дней",
    r"наказание[^.]*?в виде\s+([а-яёА-ЯЁ]+)\s+лет\s+([а-яёА-ЯЁ]+)\s+месяцев\s+и\s+([а-яёА-ЯЁ]+)\s+дней",

    r"наказание[^.]*?в виде\s+([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+месяц",
    r"назначить[^.]*?([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+месяц",
    r"срок\s+([а-яёА-ЯЁ]+)\s+лет\s+и\s+([а-яёА-ЯЁ]+)\s+месяц",
    r"наказание[^.]*?в виде\s+([а-яёА-ЯЁ]+)\s+лет",
    r"назначить[^.]*?([а-яёА-ЯЁ]+)\s+лет",
    r"срок\s+([а-яёА-ЯЁ]+)\s+лет",

    r"наказание[^.]{0,100}?в виде[^.]{0,100}?лишения\s+свободы[^.]{0,100}?на\s+срок\s+(\d+)[^0-9а-яёА-ЯЁ]{0,10}(?:лет|года|год)[^0-9а-яёА-ЯЁ]{0,10}(\d+)?[^0-9а-яёА-ЯЁ]{0,10}(?:дней|дня)?",

    r"наказание[^.]{0,100}?в виде[^.]{0,100}?лишения\s+свободы[^.]{0,100}?на\s+срок\s+(\d+)[^0-9а-яёА-ЯЁ]{0,10}(?:лет|года|год)[^0-9а-яёА-ЯЁ]{0,10}(\d+)?[^0-9а-яёА-ЯЁ]{0,10}(?:месяц|месяцев|месяца)?",
    r"лишения\s+свободы[^.]{0,100}?на\s+срок\s+(\d+)[^0-9а-яёА-ЯЁ]{0,10}(?:лет|года|год)[^0-9а-яёА-ЯЁ]{0,10}(\d+)?[^0-9а-яёА-ЯЁ]{0,10}(?:месяц|месяцев|месяца)?",
    r"назначить[^.]{0,100}?лишения\s+свободы[^.]{0,100}?на\s+срок\s+(\d+)[^0-9а-яёА-ЯЁ]{0,10}(?:лет|года|год)[^0-9а-яёА-ЯЁ]{0,10}(\d+)?[^0-9а-яёА-ЯЁ]{0,10}(?:месяц|месяцев|месяца)?",
    r"(?:сроком\s+на|в\s+виде\s+лишения\s+свободы\s+на\s+)?(\d+)[^0-9а-яёА-ЯЁ]{0,10}(?:лет|года|год)[^0-9а-яёА-ЯЁ]{0,10}(\d+)?[^0-9а-яёА-ЯЁ]{0,10}(?:месяц|месяцев|месяца)?",

    r"В ВИДЕ\s+([А-ЯЁ]+)\s+ЛЕТ",
    r"НАЗНАЧИТЬ\s+([А-ЯЁ]+)\s+ЛЕТ"
]

def extract_prison_term(text):
    text_clean = clean_text(text)
    text_lower = text_clean.lower()

    if "пожизн" in text_lower:
        return 100.0

    last_total = None

    for pattern in prison_term_patterns:
        try:
            for match in re.finditer(pattern, text_clean, re.IGNORECASE):
                groups = match.groups()
                if len(groups) >= 1:
                    if len(groups) >= 2 and ("дней" in text_lower or "дня" in text_lower or "день" in text_lower):
                        years = groups[0]
                        days = groups[1]
                        total = extract_number_with_months(years, None, days)
                        if total > 0:
                            last_total = round(total, 4)
                    else:
                        years = groups[0]
                        months = groups[1] if len(groups) > 1 else None
                        total = extract_number_with_months(years, months)
                        if total > 0:
                            last_total = round(total, 2)
        except re.error:
            continue

    if last_total is not None:
        return last_total
    explicit_num_matches = list(re.finditer(r"(?:назначить|наказание|срок)[^.]*?(\d+)\s*(?:лет|года|год)", text_clean, re.IGNORECASE))
    if explicit_num_matches:
        last_match = explicit_num_matches[-1]
        return float(last_match.group(1))

    return None

# Пример использования
text = """
П Р И Г О В О Р И Л :
признать АНДРЕЕВА В.Д. виновным в совершении преступления, предусмотренного ч. 1 ст. 105 УК РФ и назначить ему наказание в виде СЕМИ ЛЕТ лишения свободы.
На основании ч. 4, ч. 5 ст. 69, п. «г» ч. 1 ст. 71 УК РФ, по совокупности преступлений, путем частичного сложения наказания, назначенного по настоящему приговору и основного наказания, назначенного по приговору Кировского районного суда Санкт-Петербурга от 13 декабря 2021 года, с присоединением дополнительного наказания поданному приговору, окончательно назначить АНДРЕЕВУ В.Д. наказание в виде СЕМИ ЛЕТ ДЕСЯТИ ДНЕЙ лишения свободы с лишением права заниматься деятельностью, связанной с управлением транспортными средствами на срок два года один месяц двадцать пять дней.
"""

result = extract_prison_term(text)
if result is not None:
    print(f"Извлеченный срок: {result} лет")
else:
    print("Срок наказания не найден")

Извлеченный срок: 2021.0 лет


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def calculate_metrics(df, threshold=0.1):
    """
    Расчет метрик качества с учетом порога точности

    Параметры:
    - df: DataFrame с колонками 'prison_term' и 'prison_term_re'
    - threshold: максимально допустимая разница (в годах) для считания предсказания верным
    """

    df_clean = df.dropna(subset=['prison_term', 'prison_term_re']).copy()
    df_clean['prison_term'] = pd.to_numeric(df_clean['prison_term'], errors='coerce')
    df_clean['prison_term_re'] = pd.to_numeric(df_clean['prison_term_re'], errors='coerce')
    df_clean = df_clean.dropna(subset=['prison_term', 'prison_term_re'])

    if len(df_clean) == 0:
        print("Нет данных для сравнения")
        return

    y_true = df_clean['prison_term']
    y_pred = df_clean['prison_term_re']

    y_true_binary = (abs(y_true - y_pred) > threshold).astype(int)
    y_pred_binary = [0] * len(y_true_binary)  # Все предсказания считаем "правильными" (0)

    # Расчет метрик
    acc = accuracy_score(y_true_binary, y_pred_binary)
    f1 = f1_score(y_true_binary, y_pred_binary, pos_label=0)  # F1 для "правильных" предсказаний

    print(f"Метрики при пороге {threshold} лет:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score: {f1:.4f}")

    # Дополнительная статистика
    exact_matches = (abs(y_true - y_pred) <= threshold).mean()
    print(f"\nДоля предсказаний с ошибкой ≤ {threshold} лет: {exact_matches:.2%}")

    diff = abs(y_pred - y_true)
    print(f"\nСтатистика по абсолютным ошибкам:")
    print(f"Средняя: {diff.mean():.4f} лет")
    print(f"Медианная: {diff.median():.4f} лет")
    print(f"Максимальная: {diff.max():.4f} лет")

    # Примеры ошибок
    large_errors = df_clean[abs(df_clean['prison_term'] - df_clean['prison_term_re']) > threshold]
    if not large_errors.empty:
        print("\nПримеры ошибок:")
        print(large_errors[['prison_term', 'prison_term_re', 'sentence']].head(3).to_string(index=False))

# Пример вызова с порогом 0.1 года
calculate_metrics(dff, threshold=0.1)

# Можно также попробовать с более строгим порогом
calculate_metrics(dff, threshold=0.5)

KeyError: ['prison_term_re']

In [ ]:
sampled_df["prison_term"] = sampled_df["sentence"].apply(extract_prison_term)

In [ ]:
sampled_df.to_csv("sampled_data.csv", index=False)

In [ ]:
import re
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

# Словарь для преобразования слов в числа с учетом всех падежей и форм
word2num = {
    # Основные числительные
    "ноль": 0, "один": 1, "одного": 1, "одна": 1, "одной": 1, "одну": 1,
    "два": 2, "двух": 2, "две": 2, "три": 3, "трех": 3, "четыре": 4,
    "пять": 5, "шесть": 6, "семь": 7, "восемь": 8, "восьми": 8,
    "девять": 9, "десять": 10,

    # Составные числительные 11-19
    "одиннадцать": 11, "двенадцать": 12, "тринадцать": 13,
    "четырнадцать": 14, "пятнадцать": 15, "шестнадцать": 16,
    "семнадцать": 17, "восемнадцать": 18, "девятнадцать": 19,

    # Десятки
    "двадцать": 20, "тридцать": 30, "сорок": 40, "пятьдесят": 50,
    "шестьдесят": 60, "семьдесят": 70, "восемьдесят": 80,
    "девяносто": 90, "сто": 100,

    # Дробные и особые формы
    "полтора": 1.5, "полторы": 1.5, "два с половиной": 2.5,
    "три с половиной": 3.5, "четыре с половиной": 4.5
}

# Дублируем словарь для верхнего регистра
word2num_upper = {k.upper(): v for k, v in word2num.items()}
word2num.update(word2num_upper)

def clean_text(text):
    """Улучшенная очистка текста с сохранением ключевых терминов"""
    if not isinstance(text, str):
        return ""

    # Удаляем текст в скобках и кавычках
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'[«»"\'`]', '', text)

    # Нормализуем пробелы и переносы строк
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)

    # Приводим к нижнему регистру, но сохраняем ключевые термины
    text = re.sub(
        r'\b(назначить|наказание|срок|лет|год|года|месяц|месяца|месяцев|день|дня|дней)\b',
        lambda m: m.group(0).lower(),
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

def extract_number(value):
    """Улучшенное извлечение числа из строки с валидацией"""
    if value is None:
        return 0

    value = str(value).strip()

    # Обработка диапазонов (например, "10-12 лет")
    if '-' in value:
        parts = value.split('-')
        if all(p.strip().isdigit() for p in parts):
            nums = [int(p.strip()) for p in parts]
            return sum(nums) / len(nums)  # Возвращаем среднее значение

    # Прямое число
    if value.isdigit():
        num = int(value)
        return num if 0 < num <= 100 else 0  # Фильтр нереалистичных значений

    # Число прописью
    lower_val = value.lower()
    num = word2num.get(lower_val, 0)

    # Валидация извлеченного числа
    return num if 0 < num <= 100 else 0

def extract_number_with_months(years_str, months_str=None, days_str=None):
    """Преобразует годы, месяцы и дни в общее количество лет"""
    years = extract_number(years_str)
    months = extract_number(months_str) if months_str else 0
    days = extract_number(days_str) if days_str else 0

    # Расчет общего срока с проверкой на валидность
    if years == 0:
        return 0

    total = years + months/12 + days/365
    return round(min(total, 100), 2)  # Ограничиваем максимальный срок 100 годами

# Улучшенные паттерны для поиска сроков
prison_term_patterns = [
    # Полные формы с годами, месяцами и днями
    r"(?:назначить|наказание|приговорить|срок)[^.]*?в виде\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s+((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?\s+((?:\d+|[\wЁё]+)\s*(?:день|дня|дней))?",
    r"лишения\s+свободы[^.]*?на\s+срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:день|дня|дней))?",

    # Формы с годами и месяцами
    r"(?:назначить|наказание)[^.]*?((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s+и\s+((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))",
    r"срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))\s*(?:и\s+)?((?:\d+|[\wЁё]+)\s*(?:месяц|месяца|месяцев))?",

    # Простые формы только с годами
    r"(?:назначить|наказание|приговорить)[^.]*?((?:\d+|[\wЁё]+)\s*(?:лет|года|год))",
    r"лишения\s+свободы[^.]*?на\s+срок\s+((?:\d+|[\wЁё]+)\s*(?:лет|года|год))",

    # Формы для верхнего регистра (капс)
    r"В\s+ВИДЕ\s+([А-ЯЁ]+)\s+ЛЕТ\s+([А-ЯЁ]+)?\s*МЕСЯЦ(?:ЕВ|А)?\s*([А-ЯЁ]+)?\s*ДЕН(?:Ь|Я|ЕЙ)?",
    r"НАЗНАЧИТЬ\s+([А-ЯЁ]+)\s+ЛЕТ\s+([А-ЯЁ]+)?\s*МЕСЯЦ(?:ЕВ|А)?",

    # Числовые формы с разделителями
    r"(?:сроком\s+на|на\s+срок)\s+(\d+)\s*(?:лет|года|год)\s*(?:и\s+)?(\d+)?\s*(?:месяц|месяца|месяцев)?\s*(?:и\s+)?(\d+)?\s*(?:день|дня|дней)?"
]

def extract_prison_term(text):
    """Основная функция для извлечения срока наказания с улучшенной обработкой"""
    try:
        if not text or not isinstance(text, str):
            return None

        text_clean = clean_text(text)
        text_lower = text_clean.lower()

        # Проверка на пожизненное заключение
        if any(word in text_lower for word in ["пожизн", "пожизненн"]):
            return 100.0

        # Поиск по всем паттернам
        for pattern in prison_term_patterns:
            matches = re.finditer(pattern, text_clean, re.IGNORECASE)
            for match in matches:
                groups = match.groups()
                if not groups or not groups[0]:
                    continue

                # Извлечение компонентов срока
                years = groups[0]
                months = groups[1] if len(groups) > 1 and groups[1] else None
                days = groups[2] if len(groups) > 2 and groups[2] else None

                # Преобразование в общий срок
                total = extract_number_with_months(years, months, days)
                if total > 0:
                    return total

        # Дополнительная проверка для простых случаев
        simple_match = re.search(
            r'(?:срок|наказание|назначить)[^.]*?(\d+)\s*(?:лет|года|год)',
            text_clean
        )
        if simple_match:
            return float(simple_match.group(1))

    except Exception as e:
        print(f"Ошибка при обработке текста: {str(e)}")
        return None

    return None

def calculate_metrics(df, true_col='prison_term', pred_col='prison_term_re', thresholds=[0.1, 0.5]):
    """
    Улучшенный расчет метрик качества с несколькими порогами

    Параметры:
    - df: DataFrame с данными
    - true_col: название колонки с истинными значениями
    - pred_col: название колонки с предсказанными значениями
    - thresholds: список порогов для оценки
    """
    # Очистка данных
    df_clean = df.dropna(subset=[true_col, pred_col]).copy()
    df_clean[true_col] = pd.to_numeric(df_clean[true_col], errors='coerce')
    df_clean[pred_col] = pd.to_numeric(df_clean[pred_col], errors='coerce')
    df_clean = df_clean.dropna(subset=[true_col, pred_col])

    if df_clean.empty:
        print("Нет данных для сравнения")
        return

    y_true = df_clean[true_col]
    y_pred = df_clean[pred_col]
    errors = abs(y_true - y_pred)

    for threshold in thresholds:
        # Бинаризация ошибок
        correct = (errors <= threshold).astype(int)

        # Расчет метрик
        acc = accuracy_score(correct, [1] * len(correct))  # Все предсказания считаем правильными
        f1 = f1_score(correct, [1] * len(correct))

        print(f"\nМетрики при пороге {threshold} лет:")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1-score: {f1:.4f}")
        print(f"Доля правильных предсказаний: {correct.mean():.2%}")

    # Дополнительная статистика
    print("\nСтатистика по абсолютным ошибкам:")
    print(f"Средняя ошибка: {errors.mean():.4f} лет")
    print(f"Медианная ошибка: {errors.median():.4f} лет")
    print(f"Максимальная ошибка: {errors.max():.4f} лет")
    print(f"Минимальная ошибка: {errors.min():.4f} лет")

    # Примеры ошибок
    large_errors = df_clean[errors > max(thresholds)]
    if not large_errors.empty:
        print("\nПримеры ошибочных предсказаний:")
        print(large_errors[[true_col, pred_col, 'sentence']].head(3).to_string(index=False))

# Пример использования
if __name__ == "__main__":
    # Тестовый текст
    test_text = """
    П Р И Г О В О Р И Л:
    признать виновным в совершении преступления и назначить наказание
    в виде ДЕСЯТИ ЛЕТ ВОСЬМИ МЕСЯЦЕВ лишения свободы.
    """

    # Извлечение срока
    term = extract_prison_term(test_text)
    print(f"Извлеченный срок: {term} лет")

    # Для расчета метрик нужно передать DataFrame с колонками:
    # 'prison_term' - истинные значения
    # 'prison_term_re' - предсказанные значения
    # 'sentence' - тексты приговоров
    # calculate_metrics(your_dataframe)

Извлеченный срок: None лет


In [ ]:
filtered = dff[dff["prison_term_re"] == 2021.0]
print(filtered[["id", "prison_term", "prison_term_re", "sentence"]])


        id  prison_term  prison_term_re  \
79  116172          7.0          2021.0   

                                             sentence  
79  П Р И Г О В О Р И Л :\nпризнать АНДРЕЕВА В.Д. ...  


In [ ]:
threshold = 1.0
discrepancies = dff[
    (dff['prison_term'].notna()) &
    (dff['prison_term_re'].notna()) &
    (abs(dff['prison_term_re'] - dff['prison_term'])) > threshold
]

incorrect_ids = discrepancies['id'].tolist()
print(f"Найдено {len(incorrect_ids)} строк с расхождением > {threshold} лет:")
print(incorrect_ids)

if not discrepancies.empty:
    print("\nПодробная информация о расхождениях:")
    print(discrepancies[['id', 'prison_term', 'prison_term_re',
                        'difference', 'sentence']].to_string(index=False))

Найдено 0 строк с расхождением > 1.0 лет:
[]


In [ ]:
text = """окончательное наказание Смолякову И.И. назначить в виде лишения свободы на срок пятнадцать лет, без ограничения свободы"""
print(extract_prison_term(text))


15.0


In [ ]:
nan_ids = dff[dff["prison_term_re"].isna()]["id"].tolist()
print("ID строк с NaN в extracted_prison_term:", nan_ids)

ID строк с NaN в extracted_prison_term: [17008, 38497, 104279]


In [ ]:
print(" Метрики для извлечения срока наказания:")
print(f"MAE (средняя абсолютная ошибка): {mean_absolute_error(y_true, y_pred):.2f}")
print(f"R² (коэффициент детерминации): {r2_score(y_true, y_pred):.2f}")
print(f"Доля точных совпадений (до 1 знака): {(y_true == y_pred).mean():.2%}")

f1 = f1_score(y_true == y_pred, [True] * len(y_true))
print(f"F1-score (точные совпадения vs. неточные): {f1:.2f}")


📊 Метрики для извлечения срока наказания:
MAE (средняя абсолютная ошибка): 1.94
R² (коэффициент детерминации): -0.55
Доля точных совпадений (до 1 знака): 69.57%
F1-score (точные совпадения vs. неточные): 0.82


In [ ]:
NUM_WORDS = {
    'ноль': 0, 'один': 1, 'одна': 1, 'два': 2, 'две': 2, 'три': 3, 'четыре': 4,
    'пять': 5, 'шесть': 6, 'семь': 7, 'восемь': 8, 'девять': 9,
    'десять': 10, 'одиннадцать': 11, 'двенадцать': 12, 'тринадцать': 13,
    'четырнадцать': 14, 'пятнадцать': 15, 'шестнадцать': 16, 'семнадцать': 17,
    'восемнадцать': 18, 'девятнадцать': 19, 'двадцать': 20, 'тридцать': 30,
    'сорок': 40, 'пятьдесят': 50, 'шестьдесят': 60, 'семьдесят': 70,
    'восемьдесят': 80, 'девяносто': 90, 'сто': 100
}

def text2num(text):
    if not text:
        return 0
    text = text.strip().lower()
    if text in NUM_WORDS:
        return NUM_WORDS[text]
    parts = text.split()
    total = 0
    for word in parts:
        total += NUM_WORDS.get(word, 0)
    return total

def extract_number(raw):
    """Преобразует строку в число (цифра или словом)"""
    if not raw:
        return 0
    raw = raw.strip().lower()
    digit_match = re.search(r"\d+", raw)
    if digit_match:
        return int(digit_match.group())
    return text2num(raw)


# Объединение текста приговора
def get_full_text(row):
    return f"{str(row['sentence'])}"

# Паттерны для извлечения срока наказания
prison_term_patterns = [
    r"лишения свободы на срок (\d+)[\s\-]*(?:лет|года|год)(?:\s*(\d+)?\s*месяц[а-я]*)?",
    r"наказание в виде лишения свободы сроком на (\d+)[\s\-]*(?:лет|года|год)(?:\s*(\d+)?\s*месяц[а-я]*)?",
    r"назначить наказание в виде лишения свободы сроком на (\d+)[\s\-]*(?:лет|года|год)(?:\s*(\d+)?\s*месяц[а-я]*)?",
    r"в виде лишения свободы сроком на (\d+)[\s\-]*(?:лет|года|год)(?:\s*(\d+)?\s*месяц[а-я]*)?",
    r"назначить.*?наказание в виде\s*(\d+)\s*(?:лет|года|год)\s+лишения свободы",
]

# Обработка датафрейма
def extract_prison_term(text):
    text = text.lower().replace('\n', ' ').replace('\r', ' ')
    if "пожизн" in text:
        return 100.0  # условно кодируем пожизненное как 100 лет
    for pattern in prison_term_patterns:
        for match in re.finditer(pattern, text):
            years = extract_number(match.group(1))
            months = extract_number(match.group(2)) if len(match.groups()) > 1 else 0
            return round(years + months / 12, 2)
    return 0.0  # если ничего не найдено

# Применение к датафрейму
df["prison_term"] = df["sentence"].apply(extract_prison_term)

# Вывод первых строк
df[["sentence", "prison_term"]].head()


,sentence,predicted_prison_term
0,П Р И Г О В О Р И Л :\nМУСИНУ ГАЛИНУ НИКОЛАЕВН...,0.0
1,приговорил:\nпризнать виновным в совершении пр...,10.0
2,П Р И Г О В О РИ Л:\nШляхтина Бориса Юрьевича ...,20.0
3,П Р И Г О В О Р И Л :\nСАБУРОВА ДЕНИСА ВИКТОРО...,100.0
4,ПРИГОВОРИЛ:\nСутягина А.И. виновным в совершен...,0.0


In [ ]:
print(df["predicted_prison_term"].value_counts())

predicted_prison_term
0.00     3837
9.00      186
8.00      178
7.00      113
6.00       92
         ... 
11.83       1
12.17       1
3.67        1
22.00       1
13.25       1
Name: count, Length: 79, dtype: int64


In [ ]:
def count_killers(names_text: str) -> int:
    text = names_text.replace('\n', ' ').replace('\r', ' ').strip()

    people = re.split(r'\s*,\s*(?=[^,]*?\s*-\s*ст\.)', text)

    count = 0
    for person in people:
        if '105' in person:
            count += 1
    return count
df_['killer_count'] = df_['names'].apply(count_killers)


## **Mental disorder**

In [ ]:
mental_disorder_positive_if = [
    r"признан[а-я]*\s+невменяем[а-я]+",
    r"в\s+состоянии\s+невменяемости",
    r"невменяем"
    r"в\s+период\s+совершения\s+преступления\s+не\s+мог\s+осознавать",
    r"в\s+состоянии\s+психоза\s+и\s+не\s+мог"
]

mental_disorder_negative_if = [
    r"признан[а-я]*\s+вменяем[а-я]+",
    r"в\s+период\s+совершения\s+преступления\s+осознавал[а-я]*\s+фактический\s+характер",
    r"мог\s+руководить\s+своими\s+действиями",
    r"вменяем[а-я]*\s+и\s+не\s+нуждается\s+в\s+принудительном",
    r"не\s+обнаружено\s+психических\s+расстройств"
    r"вменяем[а-я]*\s+и\s+не\s+нуждается\s+в\s+принудительном",
    r"(не\s+определялись\s+какие-либо\s+психопатологические\s+расстройства)",
    r"(действия\s+носили\s+последовательный,\s+целенаправленный\s+характер)",
    r"(сохранялся\s+адекватный\s+смысловой\s+и\s+речевой\s+контакт)",

    r"(могла\s+в\s+полной\s+мере\s+осознавать\s+фактический\s+характер\s+и\s+общественную\s+опасность)",
    r"(не\s+нуждается\s+в\s+применении\s+принудительных\s+мер\s+медицинского\s+характера)",
    r"(сохранность\s+интеллектуально-мнестических\s+функций)",
    r"(сохранность\s+критических\s+и\s+прогностических\s+способностей)"
]

def check_mental_disorder(text):
    if not isinstance(text, str) or pd.isna(text):
        return "нет"

    text = text.lower()
    strong_no_evidence = any(re.search(pattern, text) for pattern in [
        r"не\s+страдает\s+(хроническим|временным)\s+психическим\s+расстройством,\s+слабоумием,\s+иным\s+болезненным\s+состоянием\s+психики",
        r"не\s+определялись\s+какие-либо\s+психопатологические\s+расстройства",
        r"в\s+применении\s+принудительных\s+мер\s+медицинского\s+характера\s+не\s+нуждается"
    ])

    if strong_no_evidence:
        return "нет
    has_disorder = any(re.search(pattern, text) for pattern in mental_disorder_patterns)

    personality_only = any(re.search(pattern, text) for pattern in [
        r"эмоциональное\s+лабильное\s+расстройство\s+личности",
        r"употребление\s+психоактивных\s+веществ",
        r"характерологические\s+особенности"
    ])
    if personality_only and not has_disorder:
        return "нет"
    return "да" if has_disorder else "нет"
df_final_100["mental_disorder_pred"] = df_final_100["description"].apply(check_mental_disorder)

valid_data = df_final_100.copy()
valid_data = valid_data.dropna(subset=['description'])
valid_data = valid_data[valid_data['mental_disorder'].isin(['да', 'нет'])]

if len(valid_data) == 0:
    print("Нет данных для расчета метрик - все строки содержат невалидные значения")
else:
    y_true = valid_data['mental_disorder'].map({"да": 1, "нет": 0})
    y_pred = valid_data['mental_disorder_pred'].map({"да": 1, "нет": 0})

    try:
        metrics = {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1-score": f1_score(y_true, y_pred, zero_division=0)
        }
        print(f"Метрики рассчитаны на {len(valid_data)} валидных записях:")
        for name, value in metrics.items():
            print(f"{name}: {value:.4f}")
        errors = valid_data[valid_data['mental_disorder'] != valid_data['mental_disorder_pred']]
        print(f"\nКоличество ошибок: {len(errors)}")
        if not errors.empty:
            print("\nПримеры ошибок (первые 3):")
            print(errors[['description', 'mental_disorder', 'mental_disorder_pred']].head(3).to_string(index=False))
            false_positives = errors[errors['mental_disorder_pred'] == 'да']
            false_negatives = errors[errors['mental_disorder_pred'] == 'нет']

            print(f"\nЛожные срабатывания (False Positives): {len(false_positives)}")
            print(f"Пропущенные случаи (False Negatives): {len(false_negatives)}")

    except ValueError as e:
        print(f"Ошибка при расчете метрик: {str(e)}")

Метрики рассчитаны на 97 валидных записях:
Accuracy: 1.0000
Precision: 0.0000
Recall: 0.0000
F1-score: 0.0000

Количество ошибок: 0


## **Time_of_day**

In [29]:
def extract_time_of_day(text):
    if not isinstance(text, str):
        return None
    time_patterns = [
        (r'(?:в|с)\s+(\d{1,2})(?:-|\s*до\s*)(\d{1,2})?\s*(?:час|ч)', 'range'),  # "в 16-18 часов"
        (r'(\d{1,2})\s*:\s*(\d{2})', 'exact'),  # "в 18:30"
        (r'в\s+(\d{1,2})\s*час', 'exact_hour'),  # "в 16 часов"
        (r'с\s+(\d{1,2})\s*до\s+(\d{1,2})', 'range'),  # "с 21 до 22 часов"
        (r'около\s+(\d{1,2})\s*(?:час|ч)', 'approx')  # "около 22 часов"
    ]

    found_times = []

    for pattern, pattern_type in time_patterns:
        matches = re.finditer(pattern, text)
        for match in matches:
            if pattern_type == 'exact':
                hour = int(match.group(1))
                found_times.append(hour)
            elif pattern_type == 'exact_hour':
                hour = int(match.group(1))
                found_times.append(hour)
            elif pattern_type == 'approx':
                hour = int(match.group(1))
                found_times.append(hour)
            elif pattern_type == 'range':
                start = int(match.group(1))
                end = int(match.group(2)) if match.group(2) else start
                found_times.extend([start, end])
    time_categories = []
    for hour in found_times:
        if 4 <= hour <= 11:
            time_categories.append('утро')
        elif 12 <= hour <= 15:
            time_categories.append('день')
        elif 16 <= hour <= 22:
            time_categories.append('вечер')
        elif hour >= 23 or hour <= 3:
            time_categories.append('ночь')

    if time_categories:
        return max(set(time_categories), key=time_categories.count)

    keyword_patterns = [
        (r'\bутр(ом|а)\b', 'утро'),
        (r'\bдн(ём|я)\b', 'день'),
        (r'\bвечер(ом|а)\b', 'вечер'),
        (r'\bноч(ью|и)\b', 'ночь'),
        (r'\bранним\s+утр(ом|а)\b', 'утро'),
        (r'\bпоздним\s+вечер(ом|а)\b', 'вечер'),
        (r'\bглубокой\s+ноч(ью|и)\b', 'ночь')
    ]

    for pattern, category in keyword_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return category

    return None

text = "в период с 21-00 часа до 22 часов 20 минут ДД.ММ.ГГГГ, Лесникова С.А., находясь в указанной квартире..."
print(extract_time_of_day(text)) 
df_final_100['predicted_time'] = df_final_100['description'].apply(extract_time_of_day)


вечер


In [ ]:

mask = df_final_100[['time_of_day', 'predicted_time']].notna().all(axis=1)
df_filtered = df_final_100[mask].copy()

valid_categories = ['утро', 'день', 'вечер', 'ночь']
df_filtered = df_filtered[df_filtered['time_of_day'].isin(valid_categories) & 
                      df_filtered['predicted_time'].isin(valid_categories)]

y_true = df_filtered['time_of_day']
y_pred = df_filtered['predicted_time']

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nF1-score (micro):", f1_score(y_true, y_pred, average='micro'))

Accuracy: 0.6794871794871795

F1-score (micro): 0.6794871794871795


In [ ]:
print(dff["predicted_time_of_day"].value_counts())

predicted_time_of_day
утро     70
день     27
ночь      2
вечер     1
Name: count, dtype: int64


In [ ]:
print("Уникальные значения в time_of_day:", dff['time_of_day'].unique())
print("Уникальные значения в predicted_time:", dff['predicted_time'].unique())

Уникальные значения в time_of_day: ['неизвестно' 'ночь' 'вечер' 'утро' "['ночь', 'утро']" "['вечер', 'ночь']"
 'день' "['утро', 'день']" "['день', 'вечер']"]
Уникальные значения в predicted_time: [None 'ночь' 'вечер' 'утро' 'день']


In [ ]:
import ast

def parse(value):
    try:
        return ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return value

df['gender_accused'] = df['gender_accused'].apply(parse)


## **Prior_convictions**

In [ ]:
convicted_patterns = [
    r"\bбыл(а)?\s+судим(а)?\b",
    r"\bранее\s+судим(а)?\b",
    r"\bиме(ет|лся|ла)\s+(непогашенн\w+\s+)?судим(ость|ости)\b",
    r"\bпривлекался(ась)?\s+к\s+уголовной\s+ответственности\b",
    r"\bналичие\s+судимости\b",
]

not_convicted_patterns = [
    r"\bне\s+судим(а)?\b",
    r"\bранее\s+не\s+судим(а)?\b",
    r"\bне\s+име(ет|л)\s+судим(ости|ость)\b",
    r"\bне\s+привлекался(ась)?\s+к\s+уголовной\s+ответственности\b",
    r"\bсудимости\s+не\s+име(ет|л)\b",
]
def extract_conviction(row):
    combined_text = f"{row['preamble']} {row['description']}".lower()

    for pattern in not_convicted_patterns:
        if re.search(pattern, combined_text):
            return "нет"
    for pattern in convicted_patterns:
        if re.search(pattern, combined_text):
            return "да"
    return "нет"

df["prior_convictions"] = df.apply(extract_conviction, axis=1)



In [ ]:
print(df["prior_convictions"].value_counts())

prior_convictions
нет    4254
да      746
Name: count, dtype: int64


In [43]:


convicted_patterns = [
    r"\bбыл(а)?\s+судим(а)?\b",
    r"\bранее\s+судим(а)?\b",
    r"\bиме(ет|лся|ла)\s+(непогашенн\w+\s+)?судим(ость|ости)\b",
    r"\bпривлекался(ась)?\s+к\s+уголовной\s+ответственности\b",
    r"\bналичие\s+судимости\b",
]

not_convicted_patterns = [
    r"\bне\s+судим(а)?\b",
    r"\bранее\s+не\s+судим(а)?\b",
    r"\bне\s+име(ет|л)\s+судим(ости|ость)\b",
    r"\bне\s+привлекался(ась)?\s+к\s+уголовной\s+ответственности\b",
    r"\bсудимости\s+не\s+име(ет|л)\b",
]

def extract_conviction(text):
    if not isinstance(text, str):
        return -1
    
    text = text.lower()
    for pattern in not_convicted_patterns:
        if re.search(pattern, text):
            return 0
    for pattern in convicted_patterns:
        if re.search(pattern, text):
            return 1
    return 0

df_final_100["full_text"] = df_final_100["preamble"].fillna('') + " " + df_final_100["description"].fillna('')

df_final_100['predicted_conviction'] = df_final_100['full_text'].apply(extract_conviction)
df_final_100["predicted_conviction"].value_counts()
df_final_100['true_conviction'] = (df_final_100['prior_convictions'] > 0).astype(int)

valid_data = df_final_100[
    (df_final_100['true_conviction'].notna()) & 
    (df_final_100['predicted_conviction'].isin([0, 1]))
].copy()

y_true = valid_data['true_conviction']
y_pred = valid_data['predicted_conviction']

print("=== Распределение предсказаний ===")
print(valid_data['predicted_conviction'].value_counts())
print("\n=== Распределение истинных значений ===")
print(valid_data['true_conviction'].value_counts())

print("\n=== Метрики ===")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-score: {f1_score(y_true, y_pred):.4f}")

=== Распределение предсказаний ===
predicted_conviction
0    86
1    14
Name: count, dtype: int64

=== Распределение истинных значений ===
true_conviction
0    100
Name: count, dtype: int64

=== Метрики ===
Accuracy: 0.8600
F1-score: 0.0000


In [ ]:
df["prior_convictions"].value_counts()

prior_convictions
нет               77
да                20
['нет', 'нет']     2
['нет', 'да']      1
Name: count, dtype: int64

## **Precrime_argument**

In [33]:
precrime_argument_patterns = [
    r"\bссора\b",
    r"\bссор[аиые]\b",
    r"\bконфликт[а-я]*\b",
    r"\bдрака\b",
    r"\bоскорблял[аие]\b",
    r"\bв результате (личной|взаимной) неприязни\b",
    r"\bна почве (возникшей )?(личной|взаимной)? ?неприязни\b",
    r"\bна почве ссоры\b",
    r"\b(вызванной|вызванного) .*? ссор[аы]\b",
    r"\bпрепирательств[ао]?\b",
    r"\bвыяснени[ея] отношений\b"
]

noprecrime_argument_patterns = [
    r"\bссоры (не (предшествовал[аи]?|было|возникало|наблюдалось))\b",
    r"\bконфликта не (было|возникло)\b",
    r"\b(действовал|убил) внезапно\b",
    r"\b(внезапно|импульсивно) возник(ший|шее|шееся) намерение\b",
    r"\bбез (предшествующей|какой-либо) ссоры\b",
    r"\bвнезапно возникшее чувство\b",
    r"\bбез выяснения отношений\b",
    r"\bне ругались, не ссорились\b",
    r"\bсловесн[а-я]* перепалки не было\b",
    r"\bникаких конфликтов между ними не было\b"
]

def has_precrime_argument(text):
    if pd.isnull(text):
        return 0
    for pattern in noprecrime_argument_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return 0
    for pattern in precrime_argument_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return 1
    return 0

def get_full_text(row):
    if 'text' in row:
        return row['text']
    elif 'sentence' in row:
        return row['sentence']
    else:
        return ' '.join(str(v) for v in row.values if isinstance(v, str))

df_final_100['precrime_argument'] = df_final_100.apply(lambda row: has_precrime_argument(get_full_text(row)), axis=1)


In [ ]:
print(df["precrime_argument"].value_counts())

precrime_argument
0    34724
1       41
Name: count, dtype: int64


In [ ]:
mismatches = dff[dff["precrime_argument"] != dff["predicted_precrime_argument"]]
print(mismatches[["id", "precrime_argument", "predicted_precrime_argument"]])


        id precrime_argument predicted_precrime_argument
12  109467              была                         нет
19   58152               нет                        была
50   32496              была                         нет
80   53901               нет                        была
92   34315               нет                        была
93    7557              была                         нет
99    2220               нет                        была


: 

In [ ]:
df.to_csv('verdicts_with_c.csv', index=False)